# Kubeflow Pipelines と Agent Platform を使用した継続的トレーニング

**学習目標:**
1. Kubeflow（KF）の事前構築済みコンポーネントの使用方法を学ぶ
1. KF 軽量 Python コンポーネントの使用方法を学ぶ
1. これらのコンポーネントを使用して KF パイプラインを構築する方法を学ぶ
1. KF パイプラインをコンパイル、アップロード、実行する方法を学ぶ


このラボでは、**scikit-learn** モデルのトレーニング、チューニング、デプロイを行う **Agent Platform** サービスをオーケストレーションする KFP パイプラインを構築、デプロイ、実行します。

## セットアップ

In [ ]:
from google.cloud import aiplatform

In [ ]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

## パイプライン設計の理解

パイプラインによって実装されるワークフローは、Python ベースのドメイン固有言語（DSL）を使用して定義されます。パイプラインの DSL は、以下で生成する `pipeline/pipeline.py` ファイル内に記述されます。

パイプラインの DSL は、ファイルパスや接続文字列などの環境固有の設定をハードコードしないように設計されています。これらの設定は、一連の環境変数を介してパイプラインコードに提供されます。

### トレーナーイメージのビルド

パイプラインのトレーニングステップには、カスタムトレーニングコンテナが必要です。カスタムトレーニングイメージは `trainer_image/Dockerfile` で定義されています。

In [ ]:
!cat trainer_image/Dockerfile

それでは、このトレーナーコンテナをビルドして Artifact Registry にプッシュしましょう。

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype"
IMAGE_TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

In [ ]:
!gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image

モデルのサービング時にトレーニング時と同じ ML フレームワークのバージョンを一致させるために、以下のサービングコンテナをパイプラインに提供する必要があります。

In [ ]:
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)

**注:** トレーニング ML フレームワークのバージョンを変更する場合は、一致するバージョンのサービングコンテナを指定する必要があります（[予測用の事前構築済みコンテナ](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers) を参照）。

## パイプラインの構築とデプロイ

パイプラインをディスクに書き込みましょう。

### 演習

1. `pipeline/training_lightweight_component.py` 内の `train_and_deploy` 関数
1. `pipeline/tuning_lightweight_component.py` 内の `tune_hyperparameters` 関数

を実装し、以下の `pipeline.py` ファイル内の TODO を完了してください。

In [ ]:
%%writefile ./pipeline/pipeline.py
# Copyright 2021 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License"); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at

# https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS"
# BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either
# express or implied. See the License for the specific language governing
# permissions and limitations under the License.
"""Kubeflow Covertype パイプライン。"""
import os

from kfp import dsl
from training_lightweight_component import train_and_deploy
from tuning_lightweight_component import tune_hyperparameters

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT_ID = os.getenv("PROJECT_ID")
REGION = os.getenv("REGION")

TRAINING_CONTAINER_IMAGE_URI = os.getenv("TRAINING_CONTAINER_IMAGE_URI")
SERVING_CONTAINER_IMAGE_URI = os.getenv("SERVING_CONTAINER_IMAGE_URI")

TRAINING_FILE_PATH = os.getenv("TRAINING_FILE_PATH")
VALIDATION_FILE_PATH = os.getenv("VALIDATION_FILE_PATH")

MAX_TRIAL_COUNT = int(os.getenv("MAX_TRIAL_COUNT", "5"))
PARALLEL_TRIAL_COUNT = int(os.getenv("PARALLEL_TRIAL_COUNT", "5"))
THRESHOLD = float(os.getenv("THRESHOLD", "0.6"))


@dsl.pipeline(
    name="covertype-kfp-pipeline",
    description="The pipeline training and deploying the Covertype classifier",
    pipeline_root=PIPELINE_ROOT,
)
def covertype_train(
    training_container_uri: str = TRAINING_CONTAINER_IMAGE_URI,
    serving_container_uri: str = SERVING_CONTAINER_IMAGE_URI,
    training_file_path: str = TRAINING_FILE_PATH,
    validation_file_path: str = VALIDATION_FILE_PATH,
    accuracy_deployment_threshold: float = THRESHOLD,
    max_trial_count: int = MAX_TRIAL_COUNT,
    parallel_trial_count: int = PARALLEL_TRIAL_COUNT,
    pipeline_root: str = PIPELINE_ROOT,
):
    staging_bucket = f"{pipeline_root}/staging"

    tuning_op =  # TODO

    accuracy = tuning_op.outputs["best_accuracy"]

    with dsl.If(
        accuracy >= accuracy_deployment_threshold, name="deploy_decision"
    ):
        train_and_deploy_op =  # TODO

### パイプラインのコンパイル

まず、パイプラインコンパイラに渡される環境変数を定義することから始めましょう。

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

Google Cloud Storage からトレーニングデータセットと検証データセットをダウンロードします。
[ウォークスルーノートブック](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/kubeflow_pipelines/walkthrough/solutions/kfp_walkthrough_vertex.ipynb) の手順に従ってデータセットがすでに作成され Google Cloud Storage に保存されている場合は、この手順をスキップできます。

In [ ]:
%%bash
gcloud storage cp gs://asl-public-data/data/covertype/training/dataset.csv $TRAINING_FILE_PATH
gcloud storage cp gs://asl-public-data/data/covertype/validation/dataset.csv $VALIDATION_FILE_PATH

**注:** アーティファクトストアが事前に作成および適切に設定されていない場合は、Agent Platform がアクセスできるようにするために **Cloud Shell** で次のコマンドを実行する必要がある場合があります。

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")
gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com" \
    --role="roles/storage.objectAdmin"
```

#### CLI コンパイラを使用したパイプラインのコンパイル

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします。

In [ ]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

### 演習

`kfp dsl compile` コマンドラインを使用して `pipeline/pipeline.py` をコンパイルします。

In [ ]:
# TODO

**注:** Python 関数からパイプラインをコンパイルするために Python SDK を使用することもできます。

```python
compiler.Compiler().compile(
    pipeline_func=covertype_train, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプラインファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプラインパッケージのデプロイ

### 演習

`aiplatform.PipelineJob` を使用してパイプラインを Agent Platform にアップロードして実行します。

In [ ]:
# TODO

### （省略可）カスタムコンポーネントのコンパイル
他のパイプラインでカスタムコンポーネントを簡単に再利用したい場合は、YAML 形式へのコンパイルを検討してください。<br>
コンポーネントの YAML とパイプラインの YAML は異なるオブジェクトを表すことに注意してください。

In [ ]:
from kfp import compiler, components
from pipeline.training_lightweight_component import train_and_deploy
from pipeline.tuning_lightweight_component import tune_hyperparameters

compiler.Compiler().compile(
    train_and_deploy, "covertype_kfp_train_and_deploy.yaml"
)
compiler.Compiler().compile(
    tune_hyperparameters, "covertype_kfp_tune_hyperparameters.yaml"
)

コンパイルされたコンポーネントを別のパイプラインで再利用するには、`kfp.components.load_component_from_file(<YAML PATH>)` を使用して簡単に読み込みます。

In [ ]:
custom_component = components.load_component_from_file(
    "covertype_kfp_train_and_deploy.yaml"
)

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0


Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.